In [1]:
# %aimport helper, tests
# %autoreload 1

In [2]:
import collections

import helper
import numpy as np
import project_tests as tests

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (
    GRU,
    Input,
    Dense,
    TimeDistributed,
    RepeatVector,
    Bidirectional,
    Embedding,
    Dropout,
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import sparse_categorical_crossentropy
import pandas as pd
import nltk

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

/Users/glora/projects/populate-ns-lex/tf-env-final/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [3]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

## Dataset

### Load Data dummy tokenized vocab


***REAL CORPUS***

In [4]:
df_real = pd.read_csv('../dataset/REAL_Corpus/gold_translation/df_REAL_benepar_parsed_translated.csv')
df_refer = pd.read_csv('../dataset/refer/referitdataset/gold_translation/df_referit_benepar_parsed_translated.csv')
df_asian_treebank = pd.read_csv('../dataset/AsianTreebank/data/asian_treebank_train.csv')
print('Dataset Loaded')

Dataset Loaded


In [5]:
ast_sentences_id = df_asian_treebank['id']
ast_sentences_en = df_asian_treebank['en']

real_sentences_en = df_real['annotation']
real_sentences_id = df_real['gold_translation']

refer_sentence_en = df_refer['annotation']
refer_sentence_id = df_refer['gold_translation']

In [ ]:
# sampled_pairs of sentences in df_asian_treebank
sampled_pairs = df_asian_treebank[['en','id']].sample(n=1185, random_state=42)
sampled_pairs = pd.DataFrame(sampled_pairs)
input_sentence_en = sampled_pairs['en']
input_sentence_id = sampled_pairs['id']

In [9]:
input_sentence_en.iloc[5]

'Two rockets landed in the city, killing a 57-year-old woman and injuring a 24-year-old bodyguard of the Israeli Defense Minister Amir Peretz, who lives in the city.'

### Files


In [10]:
for sample_i in range(5,7):
    print('input_sentence_en Line {}:  {}'.format(sample_i + 1, input_sentence_en.iloc[sample_i]))
    print('input_sentence_id Line {}:  {}'.format(sample_i + 1, input_sentence_id.iloc[sample_i]))

input_sentence_en Line 6:  Two rockets landed in the city, killing a 57-year-old woman and injuring a 24-year-old bodyguard of the Israeli Defense Minister Amir Peretz, who lives in the city.
input_sentence_id Line 6:  Dua buah roket mendarat di kota ini, menewaskan seorang wanita berusia 57 tahun dan melukai seorang pria berusia 24 tahun yang merupakan pengawal pribadi Menteri Pertahanan Israel, Amir Peretz, yang menetap di kota itu.
input_sentence_en Line 7:  After working as assistant director on L'Aveu with Costa Gavras he directed his first film in 1973, France, Inc.
input_sentence_id Line 7:  Setelah bekerja sebagai asisten sutradara dalam L'Aveu dengan Costa Gavras dia menyutradarai film pertamanya pada tahun 1973, France, Inc.



### Vocabulary


In [11]:
english_words_counter = collections.Counter([word for sentence in input_sentence_en for word in sentence.split()])
id_words_counter = collections.Counter([word for sentence in input_sentence_id for word in sentence.split()])

print('{} English words.'.format(len([word for sentence in input_sentence_en for word in sentence.split()])))
print('{} unique English words.'.format(len(english_words_counter)))
print('10 Most common words in the English dataset:')
print('"' + '" "'.join(list(zip(*english_words_counter.most_common(10)))[0]) + '"')
print()
print('{} Indonesian words.'.format(len([word for sentence in input_sentence_id for word in sentence.split()])))
print('{} unique Indonesian words.'.format(len(id_words_counter)))
print('10 Most common words in the Indonesian dataset:')
print('"' + '" "'.join(list(zip(*id_words_counter.most_common(10)))[0]) + '"')

26592 English words.
8746 unique English words.
10 Most common words in the English dataset:
"the" "to" "of" "and" "in" "a" "that" "for" "The" "was"

24527 Indonesian words.
8089 unique Indonesian words.
10 Most common words in the Indonesian dataset:
"yang" "dan" "di" "untuk" "dari" "pada" "dengan" "bahwa" "dalam" "telah"


For comparison, _Alice's Adventures in Wonderland_ contains 2,766 unique words of a total of 15,500 words.
## Preprocess


Time to start preprocessing the data...
### Tokenize

In [12]:
def tokenize(x):
    """
    Tokenize x
    :param x: List of sentences/strings to be tokenized
    :return: Tuple of (tokenized x data, tokenizer used to tokenize x)
    """
    x_tk = Tokenizer()
    x_tk.fit_on_texts(x)
    return x_tk.texts_to_sequences(x), x_tk
tests.test_tokenize(tokenize)

# Tokenize Example output
text_tokenized, text_tokenizer = tokenize(input_sentence_en)
print(text_tokenizer.word_index)
print()
for sample_i, (sent, token_sent) in enumerate(zip(input_sentence_en[:3], text_tokenized)):
    print('Sequence {} in x'.format(sample_i + 1))
    print('  Input:  {}'.format(sent))
    print('  Output: {}'.format(token_sent))

{'the': 1, 'to': 2, 'of': 3, 'and': 4, 'in': 5, 'a': 6, 'that': 7, 'for': 8, 'was': 9, 'on': 10, 'is': 11, 'by': 12, 'with': 13, 'at': 14, 'as': 15, 'it': 16, 'have': 17, 'from': 18, 'he': 19, 'has': 20, 'said': 21, 'be': 22, 'were': 23, 'been': 24, 'his': 25, 'are': 26, 'an': 27, 'this': 28, 'not': 29, 'will': 30, 'had': 31, 'they': 32, 'which': 33, 'people': 34, 'their': 35, 'who': 36, 'after': 37, 'but': 38, 'first': 39, 'i': 40, 'new': 41, 'also': 42, 'two': 43, 'time': 44, 'we': 45, 'up': 46, 'one': 47, 'over': 48, 'would': 49, 'police': 50, 'when': 51, 'about': 52, 'according': 53, 'year': 54, 'minister': 55, 'no': 56, 'only': 57, 'three': 58, 'its': 59, 'there': 60, 'or': 61, 'into': 62, 'more': 63, 'government': 64, 'some': 65, 'other': 66, 'united': 67, 'all': 68, 'out': 69, 'her': 70, 'million': 71, 'could': 72, 'since': 73, 'years': 74, 'where': 75, '1': 76, 'while': 77, 'may': 78, 'being': 79, 'before': 80, 'then': 81, 'al': 82, 'u': 83, 's': 84, 'such': 85, 'during': 86, '

### Padding

Make sure all the English sequences have the same length and all the Indonesian sequences have the same length by adding padding to the **end** of each sequence using Keras's [`pad_sequences`](https://keras.io/preprocessing/sequence/#pad_sequences) function.

In [13]:
def pad(x, length=None):
    """
    Pad x
    :param x: List of sequences.
    :param length: Length to pad the sequence to.  If None, use length of longest sequence in x.
    :return: Padded numpy array of sequences
    """
    # TODO: Implement
    if length == None:
        length = max([len(sentence) for sentence in x])
    return pad_sequences(x,maxlen=length,padding='post')
tests.test_pad(pad)

# Pad Tokenized output
test_pad = pad(text_tokenized)
for sample_i, (token_sent, pad_sent) in enumerate(zip(text_tokenized[:3], test_pad)):
    print('Sequence {} in x'.format(sample_i + 1))
    print('  Input:  {}'.format(np.array(token_sent)))
    print('  Output: {}'.format(pad_sent))

Sequence 1 in x
  Input:  [ 111 1143   17  550   56 2684    2 1623    5  189  849  157    1  338
    8 2685 2686   52  256 2687 2688]
  Output: [ 111 1143   17  550   56 2684    2 1623    5  189  849  157    1  338
    8 2685 2686   52  256 2687 2688    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0]
Sequence 2 in x
  Input:  [ 551    9  339   12 1144    2 1145 2689  133    4 1146    5  257    2
    1  340   14  234  677  258   15    6 2690]
  Output: [ 551    9  339   12 1144    2 1145 2689  133    4 1146    5  257    2
    1  340   14  234  677  258   15    6 2690    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0]
Sequence 3 in x
  Input:  [   1   58 269

### Preprocess Pipeline

In [14]:
def preprocess(x, y):
   
    preprocess_x, x_tk = tokenize(x)
    preprocess_y, y_tk = tokenize(y)

    preprocess_x = pad(preprocess_x)
    preprocess_y = pad(preprocess_y)

    # Keras's sparse_categorical_crossentropy function requires the labels to be in 3 dimensions
    preprocess_y = preprocess_y.reshape(*preprocess_y.shape, 1)

    return preprocess_x, preprocess_y, x_tk, y_tk

preproc_sentence_en, preproc_sentence_id, en_tokenizer, id_tokenizer =\
    preprocess(input_sentence_en, input_sentence_id)
    
max_english_sequence_length = preproc_sentence_en.shape[1]
max_indo_sequence_length = preproc_sentence_id.shape[1]
english_vocab_size = len(en_tokenizer.word_index)
indonesian_vocab_size = len(id_tokenizer.word_index)

print('Data Preprocessed')
print("Max English sentence length:", max_english_sequence_length)
print("Max indonesian sentence length:", max_indo_sequence_length)
print("English vocabulary size:", english_vocab_size)
print("Indonesian vocabulary size:", indonesian_vocab_size)

Data Preprocessed
Max English sentence length: 65
Max indonesian sentence length: 62
English vocabulary size: 6576
Indonesian vocabulary size: 6009


## Models

- Model 1 is a simple RNN
- Model 2 is a RNN with Embedding
- Model 3 is a Bidirectional RNN
- Model 4 is an optional Encoder-Decoder RNN


### Ids Back to Text


In [15]:
def logits_to_text(logits, tokenizer):
   
    index_to_words = {id: word for word, id in tokenizer.word_index.items()}
    index_to_words[0] = '<PAD>'

    return ' '.join([index_to_words[prediction] for prediction in np.argmax(logits, 1)])

print('`logits_to_text` function loaded.')

`logits_to_text` function loaded.


### Model 5: Custom
incorporates embedding and a bidirectional rnn into one model.

In [16]:
# def model_final(input_shape, output_sequence_length, english_vocab_size, indonesian_vocab_size):
#     """
#     Build and train a model that incorporates embedding, encoder-decoder, and bidirectional RNN on x and y
#     :param input_shape: Tuple of input shape
#     :param output_sequence_length: Length of output sequeånce
#     :param english_vocab_size: Number of unique English words in the dataset
#     :param indonesian_vocab_size: Number of unique Indonesian words in the dataset
#     :return: Keras model built, but not trained
#     """
#     #Config Hyperparameters
#     learning_rate = 0.01
#     latent_dim = 128
    
#     #Config Model
#     inputs = Input(shape=input_shape[1:])
#     embedding_layer = Embedding(input_dim=english_vocab_size,
#                                 output_dim=output_sequence_length,
#                                 mask_zero=False)(inputs)
#     bd_layer = Bidirectional(GRU(output_sequence_length))(embedding_layer)
#     encoding_layer = Dense(latent_dim, activation='relu')(bd_layer)
#     decoding_layer = RepeatVector(output_sequence_length)(encoding_layer)
#     output_layer = Bidirectional(GRU(latent_dim, return_sequences=True))(decoding_layer)
#     outputs = TimeDistributed(Dense(indonesian_vocab_size, activation='softmax'))(output_layer)
    
#     #Create Model from parameters defined above
#     model = Model(inputs=inputs, outputs=outputs)
#     model.compile(
#     loss='sparse_categorical_crossentropy',
#     optimizer=Adam(1e-2),
#     # sample_weight_mode='temporal',   # tells Keras you’ll pass per-timestep weights
#     metrics=['accuracy']
# )
    
#     return model
# tests.test_model_final(model_final)
# print('Final Model Loaded')



In [17]:
# tmp_x = pad(preproc_sentence_en, max_indo_sequence_length)
# # tmp_x = tmp_x.reshape((-1, preproc_sentence_id.shape[-2], 1))

# model_bidirect_endec_emb = model_final(
#        tmp_x.shape,
#     max_indo_sequence_length,
#     english_vocab_size + 1,
#     indonesian_vocab_size + 1)
# model_bidirect_endec_emb.summary()
# mask = (preproc_sentence_id[...,0] != 0).astype('float32')  

# model_bidirect_endec_emb.fit(
#   x=tmp_x, 
#   y=preproc_sentence_id, 
#   sample_weight=mask,     # shape (N, seq_len)
#   batch_size=1024, 
#   epochs=10, 
#   validation_split=0.2
# )

In [18]:
# # 1) Define which examples to print
# example_indices = [3,4,5,6,7,9]

# # 2) Assuming you have these lists from your preprocessing step:
# #    english_sentences = [...]     
# #    indonesian_sentences = [...]  

# for ex_num, i in enumerate(example_indices, start=1):

#     # grab the raw sentences
#     input_sentence  = input_sentence_en[i]
#     target_sentence = input_sentence_id[i]

#     # model_input already has the right 3D shape
#     model_input = tmp_x[i:i+1]

#     # run inference
#     pred_logits = model_bidirect_endec_emb.predict(model_input)[0]
#     predicted_sentence = logits_to_text(pred_logits, id_tokenizer)

#     print(f"Example {ex_num}:")
#     print(f"  English (Input):     {input_sentence}")
#     print(f"  Indonesian (Target): {target_sentence}")
#     print(f"  Indonesian (Pred):   {predicted_sentence}")
#     print("-" * 60)


### Model fix aminnn

In [19]:

from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


def model_final_fix(input_shape, output_sequence_length, english_vocab_size, indonesian_vocab_size):
    """
    Fixed version of the Seq2Seq model without attention, using proper embedding dimension, masking,
    bidirectional RNNs, and dropout to reduce overfitting and handle padding correctly.
    :param input_shape: Tuple representing the shape of the input array (including batch dimension)
    :param output_sequence_length: Number of time‐steps in the output sequence
    :param english_vocab_size: Size of the English vocabulary (including the padding token)
    :param indonesian_vocab_size: Size of the Indonesian vocabulary (including the padding token)
    :return: A compiled Keras Model ready for training
    """
    # Config Hyperparameters
    embedding_dim = 128
    latent_dim = 256
    learning_rate = 0.001

    # Build Model
    inputs = Input(shape=input_shape[1:])
    embedding_layer = Embedding(
        input_dim=english_vocab_size,
        output_dim=embedding_dim,
        mask_zero=True
    )(inputs)

    # Encoder: bidirectional GRU with dropout
    encoder = Bidirectional(
        GRU(latent_dim // 2, dropout=0.3, recurrent_dropout=0.2)
    )(embedding_layer)
    encoder = Dense(latent_dim, activation='tanh')(encoder)
    encoder = Dropout(0.3)(encoder)

    decoder_inputs = Input(shape=(output_sequence_length,), name='dec_input')
    
    dec_emb = Embedding(indonesian_vocab_size, output_dim=embedding_dim, mask_zero=True, name='dec_embedding')(decoder_inputs)

    decoder_gru = GRU(
        latent_dim,
        return_sequences=True,
        dropout=0.3,
        recurrent_dropout=0.2,
        name='decoder_gru')
    
    decoder_outputs = decoder_gru(dec_emb, initial_state=encoder)
    
    # Final time‐distributed dense layer with softmax over the Indonesian vocab
    outputs = TimeDistributed(
        Dense(indonesian_vocab_size, activation='softmax')
    )(decoder_outputs)

    model = Model(inputs=[inputs, decoder_inputs], outputs=outputs)
    model.compile(
        loss='sparse_categorical_crossentropy',
        optimizer=Adam(learning_rate, clipnorm=1.0),
        # sample_weight_mode='temporal',
        metrics=['accuracy']
    )
    return model

# Test that the signature is correct
tests.test_model_final(model_final_fix)
print('Final Model Loaded')

# Prepare data
max_en_length = max(len(sentence) for sentence in preproc_sentence_en)
tmp_x = pad(preproc_sentence_en, max_en_length)

# Instantiate and inspect the model
model_bidirect_endec_emb = model_final_fix(
    tmp_x.shape,
    max_indo_sequence_length,
    english_vocab_size + 1,
    indonesian_vocab_size + 1
)
model_bidirect_endec_emb.summary()

# Create mask for padding tokens
# mask = (preproc_sentence_id[..., 0] != 0).astype('float32')

# Set up callbacks
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]

decoder_input_data = [
    [1] + sentence[:-1] for sentence in preproc_sentence_id
]

decoder_input_data = pad(decoder_input_data, length=max_indo_sequence_length)
target_data = pad(preproc_sentence_id, length=max_indo_sequence_length)  
# Train the model
model_bidirect_endec_emb.fit(
    x=[tmp_x, decoder_input_data],
    y=target_data,
    batch_size=128,
    epochs=15,
    validation_split=0.2,
    callbacks=callbacks,
    verbose=1
)


2025-06-19 12:24:01.920460: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1
2025-06-19 12:24:01.922451: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2025-06-19 12:24:01.922470: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.33 GB
2025-06-19 12:24:01.922507: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-06-19 12:24:01.922533: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Final Model Loaded


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 65)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 65, 128)   │    841,856 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_2         │ (None, 65)        │          0 │ input_layer_1[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_1     │ (None, 256)       │    198,144 │ embedding_1[0][0… │
│ (Bidirectional)     │                   │            │ not_equal_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dec_input           │ (None, 62)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 256)       │     65,792 │ bidirectional_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dec_embedding       │ (None, 62, 128)   │    769,280 │ dec_input[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 256)       │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_gru (GRU)   │ (None, 62, 256)   │    296,448 │ dec_embedding[0]… │
│                     │                   │            │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_3         │ (None, 62)        │          0 │ dec_input[0][0]   │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_1  │ (None, 62, 6010)  │  1,544,570 │ decoder_gru[0][0… │
│ (TimeDistributed)   │                   │            │ not_equal_3[0][0] │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,716,090 (14.18 MB)

 Trainable params: 3,716,090 (14.18 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/15


2025-06-19 12:24:13.968359: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


8/8 ━━━━━━━━━━━━━━━━━━━━ 1279s 156s/step - accuracy: 0.4530 - loss: 8.3638 - val_accuracy: 0.6585 - val_loss: 5.6793 - learning_rate: 0.0010
Epoch 2/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 682s 84s/step - accuracy: 0.6586 - loss: 4.8622 - val_accuracy: 0.6585 - val_loss: 3.2213 - learning_rate: 0.0010
Epoch 3/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 654s 81s/step - accuracy: 0.6612 - loss: 3.1830 - val_accuracy: 0.6585 - val_loss: 3.1776 - learning_rate: 0.0010
Epoch 4/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 650s 79s/step - accuracy: 0.6635 - loss: 3.0738 - val_accuracy: 0.6585 - val_loss: 3.1046 - learning_rate: 0.0010
Epoch 5/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 662s 84s/step - accuracy: 0.6601 - loss: 2.9580 - val_accuracy: 0.6585 - val_loss: 2.9940 - learning_rate: 0.0010
Epoch 6/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 614s 77s/step - accuracy: 0.6626 - loss: 2.8030 - val_accuracy: 0.6585 - val_loss: 2.8783 - learning_rate: 0.0010
Epoch 7/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 1366s 184s/step - accuracy: 0.6619 - loss: 2.6865 - val_accuracy: 0.6585 

In [20]:
# 1) Define which examples to print
example_indices = [3,4,5,6,7,9]

# 2) Assuming you have these lists from your preprocessing step:
#    english_sentences = [...]     
#    indonesian_sentences = [...]  

for ex_num, i in enumerate(example_indices, start=1):

    # grab the raw sentences
    input_sentence  = input_sentence_en.iloc[i]
    target_sentence = input_sentence_id.iloc[i]

    # model_input already has the right 3D shape
    model_input = tmp_x[i:i+1]

    # run inference
    pred_logits = model_bidirect_endec_emb.predict(model_input)[0]
    predicted_sentence = logits_to_text(pred_logits, id_tokenizer)

    print(f"Example {ex_num}:")
    print(f"  English (Input):     {input_sentence}")
    print(f"  Indonesian (Target): {target_sentence}")
    print(f"  Indonesian (Pred):   {predicted_sentence}")
    print("-" * 60)


ValueError: Layer "functional_1" expects 2 input(s), but it received 1 input tensors. Inputs received: [<tf.Tensor 'data:0' shape=(1, 65) dtype=int32>]

In [ ]:
def get_all_predictions(model, input_data, tokenizer):
    """
    :param model: a trained Keras model that outputs logits
    :param input_data: array of shape (N, seq_len) or (N, seq_len, 1)
    :param tokenizer: the target‐language tokenizer
    :return: list of N decoded sentences
    """
    logits = model.predict(input_data)
    return [logits_to_text(log, tokenizer) for log in logits]

# usage
all_preds_refer = get_all_predictions(model_bidirect_endec_emb, tmp_x, id_tokenizer)

38/38 ━━━━━━━━━━━━━━━━━━━━ 352s 9s/step


In [ ]:
# save the model
# model_fixed.save('bidirect_emb_final_fixed.tf')

In [ ]:
# loaded_model_final_fix = tf.keras.models.load_model('bidirect_emb_final_fixed.tf')

In [ ]:
all_preds_refer[:5]

['di di yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang',
 'di di yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang',
 'di di yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang yang',
 'di di yang yang yang yang yang yang yang yang yang yang yang yang ya

## Prediction

In [ ]:
def final_predictions(x, y, x_tk, y_tk):
    """
    Gets predictions using the final model
    :param x: Preprocessed English data
    :param y: Preprocessed Indonesian data
    :param x_tk: English tokenizer
    :param y_tk: Indonesian tokenizer
    """
    model = model_final(x.shape,
                        y.shape[1],
                       len(x_tk.word_index) + 1,
                       len(y_tk.word_index) + 1)
    model.summary()
    model.fit(x, y, batch_size=1024, epochs=10, validation_split=0.2)
    
    y_id_to_word = {value: key for key, value in y_tk.word_index.items()}
    y_id_to_word[0] = '<PAD>'

    sentence = input_sentence_en[393]
    sentence = [x_tk.word_index[word] for word in sentence.split()]
    sentence = pad_sequences([sentence], maxlen=x.shape[-1], padding='post')
    sentences = np.array([sentence[0], x[0]])
    predictions = model.predict(sentences, len(sentences))

    print('Sample 1:')
    print(' '.join([y_id_to_word[np.argmax(x)] for x in predictions[0]]))
    print('Sample 2:')
    print(' '.join([y_id_to_word[np.argmax(x)] for x in predictions[1]]))
    print(' '.join([y_id_to_word[np.max(x)] for x in y[0]]))


final_predictions(preproc_sentence_en, preproc_sentence_id, en_tokenizer, id_tokenizer)

Model: "functional_28"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_31 (InputLayer)     │ (None, 21)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_16 (Embedding)        │ (None, 21, 16)         │        13,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_17                │ (None, 32)             │         3,264 │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_37 (Dense)                │ (None, 128)            │         4,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector_9 (RepeatVector)  │ (None, 16, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_18                │ (None, 16, 256)        │       198,144 │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_28             │ (None, 16, 775)        │       199,175 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 418,199 (1.60 MB)

 Trainable params: 418,199 (1.60 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 20s 20s/step - accuracy: 1.9778e-04 - loss: 6.6514 - val_accuracy: 0.0011 - val_loss: 6.7648
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 11s 11s/step - accuracy: 0.0011 - loss: 6.7641 - val_accuracy: 0.6511 - val_loss: 3.5035
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 10s 10s/step - accuracy: 0.6595 - loss: 3.4559 - val_accuracy: 0.6511 - val_loss: 2.9055
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 11s 11s/step - accuracy: 0.6595 - loss: 2.8130 - val_accuracy: 0.6511 - val_loss: 2.5814
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 12s 12s/step - accuracy: 0.6595 - loss: 2.4627 - val_accuracy: 0.6511 - val_loss: 2.5830
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 10s 10s/step - accuracy: 0.6595 - loss: 2.4555 - val_accuracy: 0.6519 - val_loss: 2.5039
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 11s 11s/step - accuracy: 0.6600 - loss: 2.3624 - val_accuracy: 0.6511 - val_loss: 2.2951
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 11s 11s/step - accuracy: 0.6595 - loss: 2.1352 - val_accuracy: 0.6511 - val_loss: 2.3

achieved accuracy of 65%

In [ ]:
# 1) Define which examples to print
example_indices = [3,4,5,6,7,9]

# 2) Assuming you have these lists from your preprocessing step:
#    english_sentences = [...]     
#    indonesian_sentences = [...]  

for ex_num, i in enumerate(example_indices, start=1):

    # grab the raw sentences
    input_sentence  = input_sentence_en[i]
    target_sentence = input_sentence_id[i]

    # model_input already has the right 3D shape
    model_input = tmp_x[i:i+1]

    # run inference
    pred_logits = model.predict(model_input)[0]
    predicted_sentence = logits_to_text(pred_logits, id_tokenizer)

    print(f"Example {ex_num}:")
    print(f"  English (Input):     {input_sentence}")
    print(f"  Indonesian (Target): {target_sentence}")
    print(f"  Indonesian (Pred):   {predicted_sentence}")
    print("-" * 60)


array([[17, 23,  1, ..., 44,  0,  0],
       [ 5, 20, 21, ..., 51,  2, 45],
       [22,  1,  9, ..., 34,  0,  0],
       ...,
       [24,  1, 10, ..., 54,  0,  0],
       [ 5, 84,  1, ...,  0,  0,  0],
       [ 0,  0,  0, ...,  0,  0,  0]], dtype=int32)

## Evaluation

### BLEU

In [ ]:
# example of BLEU score calculation
hypothesis = ['It', 'is', 'a', 'cat', 'at', 'room']
reference = ['It', 'is', 'a', 'cat', 'inside', 'the', 'room']

BLEUscore = nltk.translate.bleu_score.sentence_bleu([reference], hypothesis)
print(BLEUscore)

0.4548019047027907


In [ ]:

smooth = SmoothingFunction().method1

bleu_scores = []
for ref, hyp in zip(sampled_pairs['id'], df_refer['predicted_translation_bidirectional']):
    reference = ref.split()
    hypothesis = hyp.split()
    score = sentence_bleu(
        [reference],
        hypothesis,
        smoothing_function=smooth,
        # you can also adjust weights for lower-order BLEU, e.g. BLEU-2:
        # weights=(0.5, 0.5, 0, 0)
    )
    bleu_scores.append(score)

average_bleu = np.mean(bleu_scores)
print(f"Average (smoothed) sentence-level BLEU for df_refer: {average_bleu:.4f}")


Average (smoothed) sentence-level BLEU for df_refer: 0.0095
